# Notebook 1 — Data Preparation
## Smart Sniper Spotter · Target Detection Module

This notebook prepares the unified person-detection training dataset by:

1. **COCO 2017** — filtering for outdoor images containing persons
2. **WiderPerson** — downloading and parsing annotations
3. **Merging** both datasets into a unified annotation format
4. **Generating TFRecord files** compatible with the TF2 Object Detection API
5. **Saving output** as a reusable Kaggle dataset for training notebooks


## 1. Environment Setup

In [ ]:
# Install required packages
!pip install -q pycocotools gdown tensorflow

import os
import json
import glob
import random
import shutil
import hashlib
from pathlib import Path
from collections import defaultdict

import numpy as np
from pycocotools.coco import COCO

print("Imports OK")


In [ ]:

COCO_ROOT = None  # Auto-detected below

# Search for COCO in /kaggle/input
for candidate in Path("/kaggle/input").iterdir():
    # Look for the annotations directory inside the candidate
    for ann_path in candidate.rglob("instances_train2017.json"):
        COCO_ROOT = str(candidate)
        break
    if COCO_ROOT:
        break

if COCO_ROOT is None:
    raise FileNotFoundError(
        "COCO 2017 not found in /kaggle/input/. "
        "Please attach a COCO 2017 dataset to this notebook. "
        "Search Kaggle datasets for 'coco-2017-dataset'."
    )

# Locate exact subdirectories
COCO_ANN_DIR = None
COCO_TRAIN_IMG_DIR = None
COCO_VAL_IMG_DIR = None

for p in Path(COCO_ROOT).rglob("instances_train2017.json"):
    COCO_ANN_DIR = str(p.parent)
for p in Path(COCO_ROOT).rglob("train2017"):
    if p.is_dir():
        COCO_TRAIN_IMG_DIR = str(p)
for p in Path(COCO_ROOT).rglob("val2017"):
    if p.is_dir():
        COCO_VAL_IMG_DIR = str(p)

print(f"COCO root:        {COCO_ROOT}")
print(f"COCO annotations: {COCO_ANN_DIR}")
print(f"COCO train imgs:  {COCO_TRAIN_IMG_DIR}")
print(f"COCO val imgs:    {COCO_VAL_IMG_DIR}")

# Working directories
WORK_DIR = "/kaggle/working"
WIDER_DIR = os.path.join(WORK_DIR, "WiderPerson")
OUTPUT_DIR = os.path.join(WORK_DIR, "snipeit_person_dataset")
TFRECORD_DIR = os.path.join(OUTPUT_DIR, "tfrecords")

os.makedirs(TFRECORD_DIR, exist_ok=True)

# Dataset parameters
RANDOM_SEED = 42
VAL_FRACTION = 0.10  # 10% of combined data for validation
TFRECORD_NUM_SHARDS_TRAIN = 16
TFRECORD_NUM_SHARDS_VAL = 4

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

print(f"\nOutput directory:  {OUTPUT_DIR}")
print(f"TFRecord dir:      {TFRECORD_DIR}")


## 2. Filter COCO 2017 for Outdoor Person Images

We extract images containing the `person` class and exclude obviously indoor scenes 
using a heuristic: if all non-person objects in an image belong to indoor-indicator 
categories (bed, couch, toilet, oven, etc.), the image is excluded.

Each annotation is stored as:
```python
{
    "image_path": str,
    "width": int,
    "height": int,
    "boxes": [(xmin, ymin, xmax, ymax), ...],  # absolute pixel coordinates
    "source": "coco"
}
```


In [ ]:
# COCO category IDs for indoor-indicator objects
# These are used to filter out obviously indoor images
INDOOR_CATEGORY_NAMES = {
    "bed", "couch", "dining table", "toilet", "tv", "laptop",
    "mouse", "remote", "keyboard", "cell phone", "microwave",
    "oven", "toaster", "sink", "refrigerator"
}

def filter_coco_person_outdoor(ann_file, img_dir):
    """
    Filter COCO for images containing persons in likely outdoor settings.
    
    Returns a list of annotation dicts with unified format.
    """
    coco = COCO(ann_file)
    
    # Get person category ID
    person_cat_ids = coco.getCatIds(catNms=["person"])
    if not person_cat_ids:
        raise ValueError("'person' category not found in COCO annotations")
    person_cat_id = person_cat_ids[0]
    
    # Get indoor category IDs
    all_cats = {c["id"]: c["name"] for c in coco.loadCats(coco.getCatIds())}
    indoor_cat_ids = {cid for cid, name in all_cats.items() if name in INDOOR_CATEGORY_NAMES}
    
    # Get all image IDs containing person
    person_img_ids = set(coco.getImgIds(catIds=person_cat_ids))
    print(f"  Total images with person: {len(person_img_ids)}")
    
    results = []
    skipped_indoor = 0
    skipped_no_box = 0
    
    for img_id in sorted(person_img_ids):
        img_info = coco.loadImgs(img_id)[0]
        img_path = os.path.join(img_dir, img_info["file_name"])
        
        if not os.path.exists(img_path):
            continue
        
        # Get all annotations for this image
        ann_ids = coco.getAnnIds(imgIds=img_id)
        anns = coco.loadAnns(ann_ids)
        
        # Check indoor heuristic: collect non-person category IDs in this image
        non_person_cats = {a["category_id"] for a in anns if a["category_id"] != person_cat_id}
        
        # If there ARE non-person objects and ALL of them are indoor → skip
        if non_person_cats and non_person_cats.issubset(indoor_cat_ids):
            skipped_indoor += 1
            continue
        
        # Extract person bounding boxes (skip crowd or tiny annotations)
        boxes = []
        for a in anns:
            if a["category_id"] != person_cat_id:
                continue
            if a.get("iscrowd", 0):
                continue
            x, y, w, h = a["bbox"]  # COCO format: [x, y, width, height]
            if w < 5 or h < 5:
                continue
            boxes.append((x, y, x + w, y + h))  # Convert to (xmin, ymin, xmax, ymax)
        
        if not boxes:
            skipped_no_box += 1
            continue
        
        results.append({
            "image_path": img_path,
            "width": img_info["width"],
            "height": img_info["height"],
            "boxes": boxes,
            "source": "coco"
        })
    
    print(f"  Kept: {len(results)} | Skipped indoor: {skipped_indoor} | No valid boxes: {skipped_no_box}")
    return results

# Process COCO train split
print("Filtering COCO train2017...")
coco_train_ann = os.path.join(COCO_ANN_DIR, "instances_train2017.json")
coco_train_data = filter_coco_person_outdoor(coco_train_ann, COCO_TRAIN_IMG_DIR)

# Process COCO val split (we'll merge and re-split later)
print("\nFiltering COCO val2017...")
coco_val_ann = os.path.join(COCO_ANN_DIR, "instances_val2017.json")
coco_val_data = filter_coco_person_outdoor(coco_val_ann, COCO_VAL_IMG_DIR)

coco_all = coco_train_data + coco_val_data
print(f"\nTotal COCO outdoor-person images: {len(coco_all)}")


## 3. Download and Parse WiderPerson

WiderPerson contains 13,382 images with ~400K person annotations across diverse 
ground-level scenarios. Annotations use the format:

```
<number of annotations>
<class_label> <x1> <y1> <x2> <y2>
...
```

Class labels: 1=pedestrian, 2=rider, 3=partially-visible, 4=ignore, 5=crowd.  
We keep classes 1, 2, and 3 as "person" and skip 4 and 5.


In [ ]:
# Download WiderPerson
# The dataset is hosted on Google Drive. If gdown fails, you can:
#   1. Download manually from: http://www.cbsr.ia.ac.cn/users/sfzhang/WiderPerson/
#   2. Upload as a private Kaggle dataset and attach it
#   3. Use an alternative mirror

import subprocess

WIDER_ZIP = os.path.join(WORK_DIR, "WiderPerson.zip")

if os.path.exists(WIDER_DIR) and len(os.listdir(WIDER_DIR)) > 0:
    print(f"WiderPerson already exists at {WIDER_DIR}, skipping download.")
else:
    # Try gdown first (Google Drive)
    # WiderPerson Google Drive file ID — update if link changes
    GDRIVE_FILE_ID = "1ZB1c9MhbhSRtB7JHrGMBs2cVr0eILfqY"
    
    print("Downloading WiderPerson from Google Drive...")
    try:
        import gdown
        gdown.download(id=GDRIVE_FILE_ID, output=WIDER_ZIP, quiet=False)
    except Exception as e:
        print(f"gdown failed: {e}")
        print("\n" + "="*70)
        print("MANUAL DOWNLOAD REQUIRED")
        print("="*70)
        print("1. Go to: http://www.cbsr.ia.ac.cn/users/sfzhang/WiderPerson/")
        print("2. Download the dataset")
        print("3. Upload as a Kaggle dataset and attach it to this notebook")
        print(f"4. Then update WIDER_DIR to point to the attached dataset path")
        print("="*70)
        raise
    
    # Extract
    if os.path.exists(WIDER_ZIP):
        print("Extracting WiderPerson...")
        !unzip -q "{WIDER_ZIP}" -d "{WORK_DIR}"
        os.remove(WIDER_ZIP)  # Free disk space
        print("Extraction complete.")
    
print(f"\nWiderPerson directory: {WIDER_DIR}")
!ls -la "{WIDER_DIR}" 2>/dev/null || echo "Directory listing failed"


In [ ]:
def parse_widerperson(data_dir):
    """
    Parse WiderPerson annotations into unified format.
    
    Returns a list of annotation dicts.
    """
    images_dir = os.path.join(data_dir, "Images")
    annotations_dir = os.path.join(data_dir, "Annotations")
    
    # Check alternative directory structures
    if not os.path.exists(images_dir):
        # Try flat structure
        candidates = list(Path(data_dir).rglob("*.jpg"))
        if candidates:
            images_dir = str(candidates[0].parent)
            print(f"  Found images at: {images_dir}")
    
    if not os.path.exists(annotations_dir):
        candidates = list(Path(data_dir).rglob("*.jpg.txt"))
        if not candidates:
            candidates = list(Path(data_dir).rglob("*.txt"))
        if candidates:
            annotations_dir = str(candidates[0].parent)
            print(f"  Found annotations at: {annotations_dir}")
    
    # Find all annotation files
    ann_files = sorted(glob.glob(os.path.join(annotations_dir, "*.jpg.txt")))
    if not ann_files:
        ann_files = sorted(glob.glob(os.path.join(annotations_dir, "*.txt")))
    
    print(f"  Found {len(ann_files)} annotation files")
    
    # Valid person class labels (1=pedestrian, 2=rider, 3=partially-visible)
    PERSON_LABELS = {1, 2, 3}
    
    results = []
    skipped = 0
    
    for ann_file in ann_files:
        basename = os.path.basename(ann_file).replace(".txt", "")
        if not basename.endswith(".jpg"):
            basename += ".jpg"
        
        img_path = os.path.join(images_dir, basename)
        
        # Try without .jpg.jpg duplication
        if not os.path.exists(img_path):
            basename_alt = os.path.basename(ann_file).replace(".jpg.txt", ".jpg")
            img_path = os.path.join(images_dir, basename_alt)
        
        if not os.path.exists(img_path):
            skipped += 1
            continue
        
        with open(ann_file, "r") as f:
            lines = f.readlines()
        
        if len(lines) < 1:
            skipped += 1
            continue
        
        try:
            num_anns = int(lines[0].strip())
        except ValueError:
            skipped += 1
            continue
        
        boxes = []
        for line in lines[1:num_anns + 1]:
            parts = line.strip().split()
            if len(parts) < 5:
                continue
            
            cls_label = int(parts[0])
            if cls_label not in PERSON_LABELS:
                continue
            
            x1, y1, x2, y2 = float(parts[1]), float(parts[2]), float(parts[3]), float(parts[4])
            
            # Sanity check
            if x2 <= x1 or y2 <= y1:
                continue
            w = x2 - x1
            h = y2 - y1
            if w < 5 or h < 5:
                continue
            
            boxes.append((x1, y1, x2, y2))
        
        if not boxes:
            skipped += 1
            continue
        
        # We need image dimensions — read from file header to avoid loading full image
        # For efficiency, we'll get dimensions during TFRecord creation
        results.append({
            "image_path": img_path,
            "width": None,   # Will be filled during TFRecord creation
            "height": None,  # Will be filled during TFRecord creation
            "boxes": boxes,
            "source": "widerperson"
        })
    
    print(f"  Kept: {len(results)} | Skipped: {skipped}")
    return results

print("Parsing WiderPerson annotations...")
wider_data = parse_widerperson(WIDER_DIR)
print(f"Total WiderPerson images: {len(wider_data)}")


## 4. Merge Datasets and Create Train/Val Split

We merge COCO outdoor-person and WiderPerson into a single pool, shuffle, 
and split into train (90%) and validation (10%) sets.


In [ ]:
# Merge all data
all_data = coco_all + wider_data
random.shuffle(all_data)

# Split
val_count = int(len(all_data) * VAL_FRACTION)
val_data = all_data[:val_count]
train_data = all_data[val_count:]

# Statistics
coco_train_count = sum(1 for d in train_data if d["source"] == "coco")
wider_train_count = sum(1 for d in train_data if d["source"] == "widerperson")
coco_val_count = sum(1 for d in val_data if d["source"] == "coco")
wider_val_count = sum(1 for d in val_data if d["source"] == "widerperson")

print(f"Combined dataset: {len(all_data)} images")
print(f"  COCO:        {len(coco_all)}")
print(f"  WiderPerson: {len(wider_data)}")
print()
print(f"Train split: {len(train_data)} images")
print(f"  COCO:        {coco_train_count}")
print(f"  WiderPerson: {wider_train_count}")
print()
print(f"Val split:   {len(val_data)} images")
print(f"  COCO:        {coco_val_count}")
print(f"  WiderPerson: {wider_val_count}")

# Count total boxes
total_train_boxes = sum(len(d["boxes"]) for d in train_data)
total_val_boxes = sum(len(d["boxes"]) for d in val_data)
print(f"\nTotal person boxes — Train: {total_train_boxes:,} | Val: {total_val_boxes:,}")


## 5. Generate TFRecord Files

TFRecords are created in the standard TF2 Object Detection API format.  
Each record contains the encoded image bytes, dimensions, and normalized 
bounding box coordinates. The records are sharded for efficient parallel 
reading during training.

**Single class:** `person` (label ID = 1)


In [ ]:
import tensorflow as tf
from PIL import Image
import io

def get_image_dimensions(image_path):
    """Get image width and height without loading full image into memory."""
    with Image.open(image_path) as img:
        return img.width, img.height

def create_tf_example(entry):
    """
    Create a tf.train.Example from an annotation entry.
    
    Returns (tf_example, success_bool).
    """
    image_path = entry["image_path"]
    
    try:
        # Read image bytes
        with open(image_path, "rb") as f:
            encoded_image = f.read()
        
        # Get dimensions if not already set
        width = entry["width"]
        height = entry["height"]
        if width is None or height is None:
            width, height = get_image_dimensions(image_path)
        
        # Determine image format
        filename = os.path.basename(image_path)
        if filename.lower().endswith(".png"):
            image_format = b"png"
        else:
            image_format = b"jpeg"
        
        # Normalize bounding boxes to [0, 1]
        xmins, xmaxs, ymins, ymaxs = [], [], [], []
        classes_text = []
        classes = []
        
        for (x1, y1, x2, y2) in entry["boxes"]:
            # Clamp to image boundaries
            x1 = max(0.0, min(float(x1), width))
            y1 = max(0.0, min(float(y1), height))
            x2 = max(0.0, min(float(x2), width))
            y2 = max(0.0, min(float(y2), height))
            
            # Normalize
            xmins.append(x1 / width)
            xmaxs.append(x2 / width)
            ymins.append(y1 / height)
            ymaxs.append(y2 / height)
            classes_text.append(b"person")
            classes.append(1)
        
        if not xmins:
            return None, False
        
        # Build tf.train.Example
        feature = {
            "image/height": tf.train.Feature(int64_list=tf.train.Int64List(value=[height])),
            "image/width": tf.train.Feature(int64_list=tf.train.Int64List(value=[width])),
            "image/filename": tf.train.Feature(bytes_list=tf.train.BytesList(value=[filename.encode("utf8")])),
            "image/source_id": tf.train.Feature(bytes_list=tf.train.BytesList(value=[filename.encode("utf8")])),
            "image/encoded": tf.train.Feature(bytes_list=tf.train.BytesList(value=[encoded_image])),
            "image/format": tf.train.Feature(bytes_list=tf.train.BytesList(value=[image_format])),
            "image/object/bbox/xmin": tf.train.Feature(float_list=tf.train.FloatList(value=xmins)),
            "image/object/bbox/xmax": tf.train.Feature(float_list=tf.train.FloatList(value=xmaxs)),
            "image/object/bbox/ymin": tf.train.Feature(float_list=tf.train.FloatList(value=ymins)),
            "image/object/bbox/ymax": tf.train.Feature(float_list=tf.train.FloatList(value=ymaxs)),
            "image/object/class/text": tf.train.Feature(bytes_list=tf.train.BytesList(value=classes_text)),
            "image/object/class/label": tf.train.Feature(int64_list=tf.train.Int64List(value=classes)),
        }
        
        example = tf.train.Example(features=tf.train.Features(feature=feature))
        return example, True
    
    except Exception as e:
        print(f"  Error processing {image_path}: {e}")
        return None, False


def write_tfrecords(data, output_prefix, num_shards):
    """Write a list of annotation entries to sharded TFRecord files."""
    
    # Create writers for each shard
    writers = []
    for i in range(num_shards):
        shard_path = f"{output_prefix}-{i:05d}-of-{num_shards:05d}.tfrecord"
        writers.append(tf.io.TFRecordWriter(shard_path))
    
    success_count = 0
    fail_count = 0
    total_boxes = 0
    
    for idx, entry in enumerate(data):
        if idx % 2000 == 0:
            print(f"  Processing {idx}/{len(data)} ({100*idx/len(data):.1f}%)")
        
        example, ok = create_tf_example(entry)
        if ok and example is not None:
            shard_idx = idx % num_shards
            writers[shard_idx].write(example.SerializeToString())
            success_count += 1
            total_boxes += len(entry["boxes"])
        else:
            fail_count += 1
    
    # Close all writers
    for w in writers:
        w.close()
    
    print(f"  Done: {success_count} examples written, {fail_count} failed")
    print(f"  Total person boxes: {total_boxes:,}")
    return success_count

print("This cell defines TFRecord generation functions. Run the next cell to generate.")


In [ ]:
# Generate train TFRecords
print("=" * 60)
print("Generating TRAIN TFRecords...")
print("=" * 60)
train_prefix = os.path.join(TFRECORD_DIR, "train")
train_count = write_tfrecords(train_data, train_prefix, TFRECORD_NUM_SHARDS_TRAIN)

print()

# Generate val TFRecords
print("=" * 60)
print("Generating VAL TFRecords...")
print("=" * 60)
val_prefix = os.path.join(TFRECORD_DIR, "val")
val_count = write_tfrecords(val_data, val_prefix, TFRECORD_NUM_SHARDS_VAL)

print(f"\nTrain TFRecords: {train_count} examples in {TFRECORD_NUM_SHARDS_TRAIN} shards")
print(f"Val TFRecords:   {val_count} examples in {TFRECORD_NUM_SHARDS_VAL} shards")


## 6. Generate Label Map

In [ ]:
# Create label map (single class: person)
LABEL_MAP_PATH = os.path.join(OUTPUT_DIR, "label_map.pbtxt")

label_map_content = """item {
  id: 1
  name: 'person'
}
"""

with open(LABEL_MAP_PATH, "w") as f:
    f.write(label_map_content)

print(f"Label map saved to: {LABEL_MAP_PATH}")
print(label_map_content)


## 7. Save Dataset Metadata

In [ ]:
# Save metadata for training notebooks
metadata = {
    "project": "Smart Sniper Spotter - Target Detection Module",
    "description": "Unified person detection dataset (COCO outdoor + WiderPerson)",
    "created_with": "Notebook 1 - Data Preparation",
    "num_classes": 1,
    "class_name": "person",
    "train_examples": train_count,
    "val_examples": val_count,
    "train_shards": TFRECORD_NUM_SHARDS_TRAIN,
    "val_shards": TFRECORD_NUM_SHARDS_VAL,
    "sources": {
        "coco_outdoor_person": len(coco_all),
        "widerperson": len(wider_data)
    },
    "train_source_breakdown": {
        "coco": coco_train_count,
        "widerperson": wider_train_count
    },
    "val_source_breakdown": {
        "coco": coco_val_count,
        "widerperson": wider_val_count
    },
    "label_map_path": "label_map.pbtxt",
    "tfrecord_dir": "tfrecords/",
    "train_pattern": "tfrecords/train-*.tfrecord",
    "val_pattern": "tfrecords/val-*.tfrecord",
    "random_seed": RANDOM_SEED,
    "val_fraction": VAL_FRACTION
}

metadata_path = os.path.join(OUTPUT_DIR, "dataset_metadata.json")
with open(metadata_path, "w") as f:
    json.dump(metadata, f, indent=2)

print(f"Metadata saved to: {metadata_path}")
print(json.dumps(metadata, indent=2))


## 8. Verify TFRecords

Quick sanity check: read a few examples from the generated TFRecords 
and verify the structure is correct.


In [ ]:
def inspect_tfrecord(tfrecord_path, num_examples=3):
    """Read and display a few examples from a TFRecord file."""
    
    feature_description = {
        "image/height": tf.io.FixedLenFeature([], tf.int64),
        "image/width": tf.io.FixedLenFeature([], tf.int64),
        "image/filename": tf.io.FixedLenFeature([], tf.string),
        "image/object/bbox/xmin": tf.io.VarLenFeature(tf.float32),
        "image/object/bbox/xmax": tf.io.VarLenFeature(tf.float32),
        "image/object/bbox/ymin": tf.io.VarLenFeature(tf.float32),
        "image/object/bbox/ymax": tf.io.VarLenFeature(tf.float32),
        "image/object/class/label": tf.io.VarLenFeature(tf.int64),
    }
    
    dataset = tf.data.TFRecordDataset(tfrecord_path)
    
    for i, raw_record in enumerate(dataset.take(num_examples)):
        parsed = tf.io.parse_single_example(raw_record, feature_description)
        
        h = parsed["image/height"].numpy()
        w = parsed["image/width"].numpy()
        fname = parsed["image/filename"].numpy().decode("utf-8")
        num_boxes = len(tf.sparse.to_dense(parsed["image/object/bbox/xmin"]).numpy())
        
        print(f"  Example {i+1}: {fname} ({w}x{h}), {num_boxes} person boxes")

# Inspect first train shard
train_shards = sorted(glob.glob(os.path.join(TFRECORD_DIR, "train-*.tfrecord")))
if train_shards:
    print(f"Inspecting {os.path.basename(train_shards[0])}:")
    inspect_tfrecord(train_shards[0])

# Inspect first val shard
val_shards = sorted(glob.glob(os.path.join(TFRECORD_DIR, "val-*.tfrecord")))
if val_shards:
    print(f"\nInspecting {os.path.basename(val_shards[0])}:")
    inspect_tfrecord(val_shards[0])


## 9. Disk Usage & Cleanup

In [ ]:
# Show output size
print("Output dataset contents:")
!du -sh {OUTPUT_DIR}/*
print()
print("TFRecord shards:")
!ls -lh {TFRECORD_DIR}/ | head -25
print()
total_tfr_size = sum(os.path.getsize(f) for f in glob.glob(os.path.join(TFRECORD_DIR, "*.tfrecord")))
print(f"Total TFRecord size: {total_tfr_size / (1024**3):.2f} GB")

# Clean up WiderPerson raw files to save space (TFRecords already contain the images)
if os.path.exists(WIDER_DIR):
    wider_size = sum(
        os.path.getsize(os.path.join(dp, f))
        for dp, dn, filenames in os.walk(WIDER_DIR)
        for f in filenames
    )
    print(f"\nWiderPerson raw files: {wider_size / (1024**3):.2f} GB")
    print("You can delete these to free disk space (TFRecords are self-contained):")
    print(f"  shutil.rmtree('{WIDER_DIR}')")


## 10. Summary & Next Steps

### What was created

| File | Description |
|------|-------------|
| `tfrecords/train-XXXXX-of-YYYYY.tfrecord` | Training data (sharded) |
| `tfrecords/val-XXXXX-of-YYYYY.tfrecord` | Validation data (sharded) |
| `label_map.pbtxt` | Single-class label map (person) |
| `dataset_metadata.json` | Dataset statistics and paths |

### How to use in training notebooks

After this notebook finishes, **save the output as a Kaggle dataset:**

1. Click **"Save Version"** → choose **"Save & Run All"**
2. After the run completes, go to the notebook's output
3. Click **"New Dataset"** → name it `snipeit-person-dataset`
4. In Notebook 2 (Training), attach this dataset

The training notebook will reference these TFRecords directly:
```python
train_pattern = "/kaggle/input/snipeit-person-dataset/snipeit_person_dataset/tfrecords/train-*.tfrecord"
val_pattern   = "/kaggle/input/snipeit-person-dataset/snipeit_person_dataset/tfrecords/val-*.tfrecord"
label_map     = "/kaggle/input/snipeit-person-dataset/snipeit_person_dataset/label_map.pbtxt"
```

### Next notebook
**Notebook 2 — Training SSD MobileNet V2 FPNLite 640×640** with full augmentation pipeline.
